In [2]:
from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import pickle
from pathlib import Path
from dotenv import load_dotenv
import os

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from scipy.signal import lfilter

env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)

MONGODB_URI = os.environ.get("MONGODB_URI")

plt.rcParams.update({'font.size': 18}) #

# Data Preprocessing

## Import Data

In [3]:
TEST_PARTICIPANT = "Grace"
PARTICIPANTS = ["CHAO", "THOMAS", "bon", "Darragh", TEST_PARTICIPANT]
CHANNELS = ["ax", "ay", "az", "gx", "gy", "gz"]
SUFFIXES = ['_l', '_r']

SAMPLING_PERIOD = 0.01 # 10ms -> 100Hz

In [4]:
client = MongoClient(MONGODB_URI)
db = client["HAR_boxing"]
collection = db["collected_data"]

mongodb_cursor = collection.find({})

In [5]:
IMU_data = []
invalid = False

for doc in mongodb_cursor:
    devices = doc["devices"]
    if len(devices) != 2: 
        continue
    
    if doc["participant"] in ["THOMAS", "CHAO"]:
        device1, device2 = list(devices.items())
        devices = {device1[0]: device2[1], device2[0]: device1[1]}

    device_data_list = []
    for device_name, device_data in devices.items():
        suffix = None
        if "Right" in device_name:
            suffix = "_r"
        elif "Left" in device_name:
            suffix = "_l"
        else:
            print("Device name does not include Left or Right")
            invalid = True
            break
    
        device_data_suffixes = {f"{k}{suffix}": v for k, v in device_data.items()}
        device_data_list.append(device_data_suffixes)

    if (invalid):
        invalid = False
        continue

    combined_device_data = device_data_list[0] | device_data_list[1]

    combined_doc = {
        'participant': doc['participant'],
        "label_timeline": doc["label_timeline"],
        **combined_device_data
    }
    IMU_data.append(combined_doc)

raw_df = pd.DataFrame(IMU_data)

In [6]:
raw_df

,participant,label_timeline,t_l,ax_l,ay_l,az_l,gx_l,gy_l,gz_l,n_l,...,t_r,ax_r,ay_r,az_r,gx_r,gy_r,gz_r,n_r,bad_packets_r,t_dev_ms_r
0,CHAO,"[{'label': 'rest', 'start': 0.0, 'end': 1.0}, ...","[0.0008, 0.0042, 0.007, 0.0477, 0.0492, 0.0507...","[-0.742, -0.709, -0.694, -0.686, -0.674, -0.66...","[-0.812, -0.788, -0.78, -0.781, -0.788, -0.793...","[-0.38, -0.381, -0.373, -0.366, -0.354, -0.337...","[-43.375, -39.0, -33.75, -29.375, -26.0, -23.5...","[6.25, 4.375, 2.5, 0.875, -0.375, -2.25, -4.5,...","[20.875, 24.5, 28.25, 30.25, 32.0, 33.75, 35.2...",302,...,"[-0.1052, -0.1028, -0.1004, -0.0572, -0.0552, ...","[-0.991, -1.012, -1.024, -1.031, -1.039, -1.04...","[0.453, 0.448, 0.452, 0.465, 0.48, 0.487, 0.49...","[-0.12, -0.122, -0.124, -0.126, -0.121, -0.112...","[-1.125, 0.0, 1.0, 1.375, -0.5, -1.0, -0.125, ...","[-30.25, -27.25, -24.375, -21.5, -15.0, -10.87...","[7.75, 6.375, 4.75, 2.75, -0.625, -3.125, -5.7...",318,0,"[0, 11, 21, 31, 45, 55, 65, 75, 85, 95, 105, 1..."
1,CHAO,"[{'label': 'rest', 'start': 0.0, 'end': 1.0}, ...","[0.0004, 0.0012, 0.003, 0.0493, 0.0501, 0.0512...","[-0.775, -0.745, -0.741, -0.741, -0.742, -0.75...","[-0.535, -0.482, -0.498, -0.53, -0.553, -0.55,...","[-0.174, -0.196, -0.204, -0.198, -0.184, -0.16...","[24.5, 19.125, 15.75, 16.0, 18.5, 20.25, 18.87...","[6.375, 6.125, 5.75, 5.5, 5.375, 5.375, 5.25, ...","[17.625, 17.625, 18.25, 19.375, 20.75, 21.875,...",338,...,"[-0.1499, -0.1487, -0.1481, -0.1011, -0.0528, ...","[-0.698, -0.727, -0.766, -0.808, -0.841, -0.87...","[0.586, 0.593, 0.598, 0.608, 0.621, 0.628, 0.6...","[-0.272, -0.292, -0.307, -0.321, -0.331, -0.33...","[-10.125, -11.25, -11.5, -10.375, -3.75, 1.875...","[-16.75, -16.0, -15.125, -14.125, -12.25, -11....","[-12.5, -10.0, -7.875, -5.75, -2.125, -1.0, -0...",320,0,"[0, 10, 20, 30, 44, 55, 65, 75, 84, 95, 105, 1..."
2,CHAO,"[{'label': 'rest', 'start': 0.0, 'end': 1.0}, ...","[0.0008, 0.0037, 0.0063, 0.0487, 0.0504, 0.052...","[-1.346, -1.341, -1.326, -1.287, -1.153, -1.08...","[-0.523, -0.5, -0.464, -0.426, -0.356, -0.297,...","[-0.164, -0.172, -0.19, -0.209, -0.223, -0.22,...","[-5.5, -4.375, -2.625, 0.125, 14.75, 26.0, 37....","[13.625, 15.0, 15.75, 16.75, 17.375, 16.375, 1...","[78.75, 75.625, 73.0, 70.25, 62.5, 58.0, 54.12...",326,...,"[-0.1058, -0.0088, -0.0066, -0.0046, -0.0019, ...","[-0.921, -0.989, -1.046, -1.112, -1.187, -1.29...","[0.306, 0.352, 0.4, 0.468, 0.578, 0.788, 0.772...","[-0.528, -0.52, -0.505, -0.491, -0.491, -0.527...","[122.5, 124.5, 125.75, 124.25, 103.875, 86.25,...","[-19.875, -20.75, -20.875, -21.375, -22.625, -...","[-87.0, -83.375, -75.875, -67.125, -47.125, -3...",306,0,"[0, 10, 20, 30, 45, 56, 66, 76, 86, 96, 105, 1..."
3,CHAO,"[{'label': 'rest', 'start': 0.0, 'end': 1.0}, ...","[0.003, 0.0057, 0.0066, 0.0505, 0.0529, 0.0542...","[-0.536, -0.538, -0.538, -0.535, -0.537, -0.54...","[-0.535, -0.57, -0.564, -0.519, -0.481, -0.498...","[-0.253, -0.258, -0.292, -0.339, -0.326, -0.28...","[114.375, 117.5, 120.0, 117.375, 99.5, 90.625,...","[-17.0, -19.0, -21.625, -25.25, -31.5, -32.5, ...","[-124.5, -127.5, -130.5, -131.5, -128.125, -12...",325,...,"[-0.0539, -0.0523, -0.05, -0.0047, -0.0015, -0...","[-0.413, -0.385, -0.359, -0.333, -0.309, -0.27...","[0.54, 0.492, 0.449, 0.401, 0.347, 0.245, 0.21...","[-0.214, -0.164, -0.136, -0.115, -0.079, -0.06...","[-19.875, -27.625, -35.0, -40.875, -44.5, -41....","[-62.875, -63.75, -64.5, -64.25, -63.25, -61.8...","[-102.125, -100.125, -96.625, -92.875, -85.75,...",314,0,"[0, 11, 21, 31, 45, 56, 66, 76, 86, 96, 106, 1..."
4,CHAO,"[{'label': 'rest', 'start': 0.0, 'end': 1.0}, ...","[-0.0433, -0.0416, -0.0395, 0.0097, 0.0132, 0....","[-0.439, -0.479, -0.521, -0.577, -0.647, -0.72...","[-1.013, -0.957, -0.919, -0.869, -0.8, -0.734,...","[0.089, 0.026, 0.006, -0.012, -0.048, -0.086, ...","[-63.625, -60.125, -57.0, -56.125, -59.375, -6...","[80.625, 79.25, 79.125, 78.375, 74.875, 68.5, ...","[178.5, 178.125, 178.125, 177.375, 175.875, 17...",329,...,"[0.0012, 0.0044, 0.0067

In [7]:
import os, pickle, itertools
from tqdm.auto import tqdm
from sklearn.base import clone

FS = 100
WIN_SIZES   = [0.5, 0.75, 1.0, 1.5]    # seconds
STRIDES     = [0.1]                     # keep fixed when comparing window sizes
MIN_PURITY  = 0.8
FILTER      = True
LP_TAU      = 0.05                      # low-pass time constant (s)
N_SEEDS     = 3
N_TREES     = 300

CACHE_DIR = "feature_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

/home/bosco/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
def to_grid(row, fs):
    t_l = np.asarray(row['t_dev_ms_l'], float)
    t_r = np.asarray(row['t_dev_ms_r'], float)
    grid = np.arange(max(t_l[0], t_r[0]), min(t_l[-1], t_r[-1]), 1000 / fs)

    sig = {}
    for suffix, t in (('_l', t_l), ('_r', t_r)):
        for ch in CHANNELS:
            sig[f'{ch}{suffix}'] = np.interp(grid, t, np.asarray(row[f'{ch}{suffix}'], float))

    labels = np.full(len(grid), None, dtype=object)
    for seg in row['label_timeline']:
        m = (grid >= seg['start'] * 1000) & (grid < seg['end'] * 1000)
        labels[m] = seg['label']
    return sig, labels


def lowpass(x, fs, tau):
    dt = 1 / fs
    a = dt / (tau + dt)
    return lfilter([a], [1, -(1 - a)], x)


def make_windows(row, record_id, win, stride, fs, min_purity, filt, tau):
    sig, labels = to_grid(row, fs)
    if filt:
        sig = {k: lowpass(v, fs, tau) for k, v in sig.items()}

    out = []
    for s in range(0, len(labels) - win + 1, stride):
        valid = labels[s:s + win][labels[s:s + win] != None]
        if len(valid) < min_purity * win:
            continue
        vals, counts = np.unique(valid, return_counts=True)
        label, purity = vals[counts.argmax()], counts.max() / win
        if purity < min_purity:
            continue

        feat = {'record_id': record_id, 'participant': row['participant'],
                'label': label, 'purity': purity, 't_start': s / fs}
        for key, x in sig.items():
            for name, v in extract_features(x[s:s + win]).items():
                feat[f'{key}_{name}'] = v
        out.append(feat)
    return out

In [24]:
import numpy as np
from scipy.stats import skew, kurtosis

def extract_features(signal):
    signal = np.asarray(signal, dtype=float)

    centred = signal - signal.mean()
    jerk = np.diff(signal)
    q = np.array_split(signal, 4)
    n = len(signal)

    return {
        'mean': signal.mean(),
        'std': signal.std(),
        'min': signal.min(),
        'max': signal.max(),
        'range': np.ptp(signal),
        'median': np.median(signal),
        'rms': np.sqrt(np.mean(signal**2)),
        'skew': skew(signal),
        'kurtosis': kurtosis(signal),
        'iqr': np.percentile(signal, 75) - np.percentile(signal, 25),
        'p10': np.percentile(signal, 10),
        'p90': np.percentile(signal, 90),
        'energy': np.sum(signal**2),
        'argmax_pos': np.argmax(signal) / n,
        'argmin_pos': np.argmin(signal) / n,
    }

In [25]:
FEATURE_CACHE = {}

def get_features(win_s, stride_s, min_purity=MIN_PURITY, filt=FILTER, tau=LP_TAU):
    key = (win_s, stride_s, min_purity, filt, tau, FS)
    if key in FEATURE_CACHE:
        return FEATURE_CACHE[key]

    path = os.path.join(CACHE_DIR, "feats_" + "_".join(map(str, key)) + ".pkl")
    if os.path.exists(path):
        df = pd.read_pickle(path)
    else:
        win, stride = int(win_s * FS), int(stride_s * FS)
        rows = []
        for rid, row in tqdm(raw_df.iterrows(), total=len(raw_df),
                             desc=f"win={win_s}s stride={stride_s}s"):
            rows.extend(make_windows(row, rid, win, stride, FS, min_purity, filt, tau))
        df = pd.DataFrame(rows)
        df.to_pickle(path)

    FEATURE_CACHE[key] = df
    return df

In [30]:
META_COLS = ['label', 'participant', 'record_id', 'purity', 't_start']

def prepare(df):
    df = df.copy()
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])
    feat_cols = [c for c in df.columns if c not in META_COLS]
    for p, g in df.groupby('participant'):
        df.loc[g.index, feat_cols] = (g[feat_cols] - g[feat_cols].mean()) / (g[feat_cols].std() + 1e-8)
    return df, feat_cols, le

LOPO_COLS = ['held_out', 'acc', 'acc_punches', 'f1_punches']
def lopo(model, data, feat_cols, le):
    rest_idx = le.transform(['rest'])[0]
    rows = []
    for p in data.participant.unique():
        tr, te = data[data.participant != p], data[data.participant == p]
        punch = (te.label != rest_idx).values
        if punch.sum() == 0 or tr.label.nunique() < 2:
            print(f"  skipping {p}: no punch windows in held-out set")
            continue
        m = clone(model).fit(tr[feat_cols], tr['label'])
        pred = m.predict(te[feat_cols])
        rows.append({'held_out': p,
                     'acc': accuracy_score(te.label, pred),
                     'acc_punches': accuracy_score(te.label[punch], pred[punch]),
                     'f1_punches': f1_score(te.label[punch], pred[punch], average='macro')})
    return pd.DataFrame(rows, columns=LOPO_COLS).set_index('held_out')

In [27]:
for win_s in WIN_SIZES:
    df = get_features(win_s, 0.1)
    counts = (df[df.label != 'rest'].groupby('participant').size()
              .reindex(df.participant.unique(), fill_value=0))
    print(f"win={win_s}s  total={len(df)}  punch windows per participant:\n{counts.to_dict()}\n")

win=0.5s  total=24171  punch windows per participant:
{'CHAO': 1440, 'THOMAS': 1440, 'bon': 1440, 'Darragh': 1407, 'Grace': 480}

win=0.75s  total=19370  punch windows per participant:
{'CHAO': 1080, 'THOMAS': 1080, 'bon': 1080, 'Darragh': 1055, 'Grace': 360}

win=1.0s  total=15638  punch windows per participant:
{'CHAO': 900, 'THOMAS': 900, 'bon': 900, 'Darragh': 878, 'Grace': 300}

win=1.5s  total=6332  punch windows per participant:
{'CHAO': 0, 'THOMAS': 0, 'bon': 0, 'Darragh': 0, 'Grace': 0}



In [31]:
# (win_s, stride_s, min_purity)
CONFIGS = [
    (0.5,  0.1, 0.8),
    (0.75, 0.1, 0.8),
    (1.0,  0.1, 0.6),
    (1.5,  0.1, 0.4),
]

all_fold_results = {}   # (win_s, stride_s, purity) -> per-participant DataFrame, averaged over seeds
summary = []

for win_s, stride_s, purity in CONFIGS:
    data, feat_cols, le = prepare(get_features(win_s, stride_s, min_purity=purity))
    rest_idx = le.transform(['rest'])[0]

    seed_results = []
    for seed in tqdm(range(N_SEEDS), desc=f"w={win_s} p={purity}", leave=False):
        rf = RandomForestClassifier(n_estimators=N_TREES, class_weight='balanced',
                                    random_state=seed, n_jobs=-1)
        seed_results.append(lopo(rf, data, feat_cols, le))

    if any(len(r) == 0 for r in seed_results):
        print(f"No usable folds for w={win_s}, purity={purity}, skipping")
        continue

    per_participant = sum(seed_results) / N_SEEDS
    all_fold_results[(win_s, stride_s, purity)] = per_participant

    summary.append({'win_s': win_s, 'stride_s': stride_s, 'min_purity': purity,
                    'n_windows': len(data),
                    'n_punch_windows': int((data.label != rest_idx).sum()),
                    'acc': per_participant.acc.mean(),
                    'acc_punches': per_participant.acc_punches.mean(),
                    'f1_punches': per_participant.f1_punches.mean(),
                    'f1_std_across_participants': per_participant.f1_punches.std()})

results_df = (pd.DataFrame(summary)
              .sort_values('f1_punches', ascending=False)
              .reset_index(drop=True)
              .round(3))
results_df

win=1.5s stride=0.1s: 100%|███████████████████████████████████████████████████████████| 776/776 [01:08<00:00, 11.37it/s]
                                                                                                                        

,win_s,stride_s,min_purity,n_windows,n_punch_windows,acc,acc_punches,f1_punches,f1_std_across_participants
0,0.75,0.1,0.8,19370,4655,0.932,0.770,0.671,0.154
1,1.00,0.1,0.6,21842,6980,0.870,0.713,0.638,0.171
2,0.50,0.1,0.8,24171,6207,0.895,0.680,0.625,0.151
3,1.50,0.1,0.4,19513,7753,0.813,0.638,0.595,0.194
